# Perceptrón de una capa: regresión lineal y regresión logística

**Materia:** Matemáticas Aplicadas a Ciencia de Datos

Este notebook es un recopilado de lo que vimos sobre un perceptron (una sola capa) para un problema de clasificación y regresión. Para más información revisen los notebooks que están en el classroom. El objetivo es mostrar que una estructura neuronal mínima, un **perceptrón de una sola capa**, permite recuperar dos modelos estadísticos conocidos:

- Regresión lineal, cuando la salida es lineal.
- Regresión logística binaria, cuando aplicamos una función sigmoide.

No estamos construyendo todavía un perceptrón multicapa. La idea es estudiar la estructura matemática básica antes de escalar a modelos con capas ocultas.

## Idea matemática central

Para un conjunto de datos organizado como:

$$X \in \mathbb{R}^{n_x \times m}$$

donde `n_x` es el número de variables y `m` el número de ejemplos, el perceptrón calcula:

$$Z = WX + b$$

con:

$$W \in \mathbb{R}^{1 \times n_x}, \qquad b \in \mathbb{R}^{1 \times 1}$$

Para regresión usamos salida identidad:

$$\hat{Y} = Z$$

Para clasificación binaria usamos salida sigmoide:

$$A = \sigma(Z) = \frac{1}{1 + e^{-Z}}$$

In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    root_mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from src.lib import (
    nn_model_regression,
    nn_model_classification,
    forward_propagation_regression,
    forward_propagation_classification,
)

## Datos

Usaremos `data/WineQT.csv`. El dataset contiene mediciones fisicoquímicas de vinos y una variable `quality`.

Usaremos el mismo conjunto de variables de entrada para dos tareas:

- **Regresión:** predecir `quality` como valor numérico.
- **Clasificación:** crear `quality_cat`, donde `1` representa vinos con `quality > 6` y `0` los demás.

En este notebook evaluaremos sobre los mismos datos usados para entrenar, porque el objetivo no es estudiar generalización sino comparar estructuras matemáticas.

In [2]:
df_wine = pd.read_csv("data/WineQT.csv")
df_wine["quality_cat"] = np.where(df_wine["quality"] > 6, 1, 0)

df_wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id,quality_cat
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4,0


In [3]:
df_wine[["quality", "quality_cat"]].describe()

,quality,quality_cat
count,1143.000000,1143.000000
mean,5.657043,0.139108
std,0.805824,0.346210
min,3.000000,0.000000
25%,5.000000,0.000000
50%,6.000000,0.000000
75%,6.000000,0.000000
max,8.000000,1.000000


## Preprocesamiento

Estandarizamos las variables numéricas para que tengan media cercana a 0 y desviación estándar cercana a 1. Esto ayuda a que el descenso del gradiente avance de manera más estable.

In [4]:
num_cols = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
    "pH", "sulphates", "alcohol",
]

target_num = "quality"
target_cat = "quality_cat"

X = df_wine.drop(columns=[target_num, target_cat])
y_num = df_wine[target_num]
y_cat = df_wine[target_cat]

preprocessor = ColumnTransformer(
    transformers=[("num", StandardScaler(), num_cols)],
    remainder="drop",
)
preprocessor.set_output(transform="pandas")

X_std = preprocessor.fit_transform(X)
X_std.columns = num_cols

X_std.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,-0.521580,0.939332,-1.365027,-0.466421,-0.231395,-0.450467,-0.363610,0.555854,1.270695,-0.573658,-0.963382
1,-0.292593,1.941813,-1.365027,0.050060,0.234247,0.915920,0.643477,0.036165,-0.708928,0.130881,-0.593601
2,-0.292593,1.273492,-1.161568,-0.171289,0.107253,-0.060071,0.246745,0.140103,-0.325775,-0.045254,-0.593601
3,1.653789,-1.399789,1.483400,-0.466421,-0.252560,0.135127,0.429852,0.659792,-0.964363,-0.456235,-0.593601
4,-0.521580,0.939332,-1.365027,-0.466421,-0.231395,-0.450467,-0.363610,0.555854,1.270695,-0.573658,-0.963382


## Forma matricial

La librería `sklearn` suele usar datos con forma `(m, n_features)`. Nuestro código manual usa la convención matemática:

$$X \in \mathbb{R}^{n_x \times m}$$

Por eso transponemos `X`.

In [5]:
X_std_array = X_std.to_numpy().T
y_num_array = y_num.to_numpy().reshape(1, len(y_num))
y_cat_array = y_cat.to_numpy().reshape(1, len(y_cat))

print("X shape:", X_std_array.shape)
print("Y regresión shape:", y_num_array.shape)
print("Y clasificación shape:", y_cat_array.shape)
print("m ejemplos:", X_std_array.shape[1])

X shape: (11, 1143)
Y regresión shape: (1, 1143)
Y clasificación shape: (1, 1143)
m ejemplos: 1143


## Parte 1: perceptrón como regresión lineal

El modelo es:

$$\hat{Y} = WX + b$$

La función de costo usada es el error cuadrático medio escalado:

$$J(W,b) = \frac{1}{2m}\sum_{i=1}^{m}(\hat{y}^{(i)} - y^{(i)})^2$$

Los parámetros se actualizan con descenso del gradiente.

In [6]:
parameters_regression = nn_model_regression(
    X_std_array,
    y_num_array,
    num_iterations=1000,
    learning_rate=0.2,
    print_cost=True,
)

W_regression = parameters_regression["W"]
b_regression = parameters_regression["b"]

print("W:", W_regression)
print("b:", b_regression)

Costo después de la iteración 0: 16.333869
Costo después de la iteración 1: 10.502000
Costo después de la iteración 2: 6.784733
Costo después de la iteración 3: 4.411713
Costo después de la iteración 4: 2.895439
Costo después de la iteración 5: 1.926019
Costo después de la iteración 6: 1.305978
Costo después de la iteración 7: 0.909283
Costo después de la iteración 8: 0.655427
Costo después de la iteración 9: 0.492945
Costo después de la iteración 10: 0.388928
Costo después de la iteración 11: 0.322325
Costo después de la iteración 12: 0.279668
Costo después de la iteración 13: 0.252338
Costo después de la iteración 14: 0.234821
Costo después de la iteración 15: 0.223587
Costo después de la iteración 16: 0.216377
Costo después de la iteración 17: 0.211744
Costo después de la iteración 18: 0.208763
Costo después de la iteración 19: 0.206840
Costo después de la iteración 20: 0.205597
Costo después de la iteración 21: 0.204790
Costo después de la iteración 22: 0.204263
Costo después de la

/Users/vanotole/Projects/Personal/mlp_exercise/src/lib.py:65: RuntimeWarning: divide by zero encountered in matmul
  Z = np.matmul(W, X) + b
/Users/vanotole/Projects/Personal/mlp_exercise/src/lib.py:65: RuntimeWarning: overflow encountered in matmul
  Z = np.matmul(W, X) + b
/Users/vanotole/Projects/Personal/mlp_exercise/src/lib.py:65: RuntimeWarning: invalid value encountered in matmul
  Z = np.matmul(W, X) + b


In [7]:
y_pred_regression = forward_propagation_regression(X_std_array, parameters_regression)

r2 = r2_score(y_num_array.ravel(), y_pred_regression.ravel())
mse = mean_squared_error(y_num_array.ravel(), y_pred_regression.ravel())
rmse = root_mean_squared_error(y_num_array.ravel(), y_pred_regression.ravel())
mae = mean_absolute_error(y_num_array.ravel(), y_pred_regression.ravel())

print(f"r2 score: {r2}")
print(f"mse: {mse}")
print(f"rmse: {rmse}")
print(f"mae: {mae}")

r2 score: 0.3742422720434323
mse: 0.40598198111590933
rmse: 0.6371671531991502
mae: 0.49465285148984817


## Comparación con `LinearRegression`

`LinearRegression` resuelve el mismo tipo de modelo lineal, pero usando herramientas numéricas internas de `sklearn` en lugar de nuestra implementación educativa de descenso del gradiente.

In [8]:
model_regression = LinearRegression()
model_regression.fit(X_std, y_num)
y_pred_regression_sklearn = model_regression.predict(X_std)

print("Betas:", model_regression.coef_)
print("Beta0:", model_regression.intercept_)
print("r2 score sklearn:", r2_score(y_num, y_pred_regression_sklearn))

Betas: [ 0.04013279 -0.20273545 -0.02592418  0.0183127  -0.08070488  0.02427669
 -0.09125551 -0.03357637 -0.06392272  0.14906773  0.30303687]
Beta0: 5.657042869641297
r2 score sklearn: 0.3742422720434534


/opt/homebrew/Caskroom/miniconda/base/envs/ds_env/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/homebrew/Caskroom/miniconda/base/envs/ds_env/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/homebrew/Caskroom/miniconda/base/envs/ds_env/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


## Parte 2: perceptrón como regresión logística

Ahora usamos la misma parte lineal:

$$Z = WX + b$$

pero la pasamos por una sigmoide:

$$A = \sigma(Z)$$

`A` se interpreta como la probabilidad de pertenecer a la clase positiva.

In [9]:
parameters_classification = nn_model_classification(
    X_std_array,
    y_cat_array,
    num_iterations=1000,
    learning_rate=1.2,
    print_cost=True,
)

W_classification = parameters_classification["W"]
b_classification = parameters_classification["b"]

print("W:", W_classification)
print("b:", b_classification)

Costo después de la iteración 0: 0.691930
Costo después de la iteración 1: 0.513278
Costo después de la iteración 2: 0.431542
Costo después de la iteración 3: 0.386234
Costo después de la iteración 4: 0.358200
Costo después de la iteración 5: 0.339519
Costo después de la iteración 6: 0.326374
Costo después de la iteración 7: 0.316726
Costo después de la iteración 8: 0.309406
Costo después de la iteración 9: 0.303702
Costo después de la iteración 10: 0.299160
Costo después de la iteración 11: 0.295478
Costo después de la iteración 12: 0.292447
Costo después de la iteración 13: 0.289920
Costo después de la iteración 14: 0.287791
Costo después de la iteración 15: 0.285979
Costo después de la iteración 16: 0.284424
Costo después de la iteración 17: 0.283081
Costo después de la iteración 18: 0.281913
Costo después de la iteración 19: 0.280891
Costo después de la iteración 20: 0.279992
Costo después de la iteración 21: 0.279198
Costo después de la iteración 22: 0.278493
Costo después de la i

/Users/vanotole/Projects/Personal/mlp_exercise/src/lib.py:88: RuntimeWarning: divide by zero encountered in matmul
  Z = np.matmul(W, X) + b
/Users/vanotole/Projects/Personal/mlp_exercise/src/lib.py:88: RuntimeWarning: overflow encountered in matmul
  Z = np.matmul(W, X) + b
/Users/vanotole/Projects/Personal/mlp_exercise/src/lib.py:88: RuntimeWarning: invalid value encountered in matmul
  Z = np.matmul(W, X) + b


In [10]:
threshold = 0.5
prob_pred_classification = forward_propagation_classification(X_std_array, parameters_classification)
y_pred_classification = np.where(prob_pred_classification > threshold, 1, 0)

accuracy = accuracy_score(y_cat_array.ravel(), y_pred_classification.ravel())
precision = precision_score(y_cat_array.ravel(), y_pred_classification.ravel())
recall = recall_score(y_cat_array.ravel(), y_pred_classification.ravel())
f1 = f1_score(y_cat_array.ravel(), y_pred_classification.ravel())

print(f"accuracy score: {accuracy}")
print(f"precision score: {precision}")
print(f"recall score: {recall}")
print(f"f1 score: {f1}")

accuracy score: 0.8836395450568679
precision score: 0.6382978723404256
recall score: 0.37735849056603776
f1 score: 0.4743083003952569


## Comparación con `LogisticRegression`

`LogisticRegression` también usa una combinación lineal y una sigmoide. La diferencia práctica está en el algoritmo de optimización y en detalles numéricos de implementación.

In [11]:
model_classification = LogisticRegression(max_iter=1000, penalty=None)
model_classification.fit(X_std, y_cat)
y_pred_classification_sklearn = model_classification.predict(X_std)

print("Betas:", model_classification.coef_)
print("Beta0:", model_classification.intercept_)
print("accuracy score:", accuracy_score(y_cat, y_pred_classification_sklearn))
print("precision score:", precision_score(y_cat, y_pred_classification_sklearn))
print("recall score:", recall_score(y_cat, y_pred_classification_sklearn))
print("f1 score:", f1_score(y_cat, y_pred_classification_sklearn))

Betas: [[ 0.30520589 -0.52168189  0.34965707  0.30392644 -0.52330414  0.07352436
  -0.40653813 -0.50475706 -0.0797371   0.57360065  0.72970281]]
Beta0: [-2.80291851]
accuracy score: 0.8836395450568679
precision score: 0.6382978723404256
recall score: 0.37735849056603776
f1 score: 0.4743083003952569


/opt/homebrew/Caskroom/miniconda/base/envs/ds_env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/homebrew/Caskroom/miniconda/base/envs/ds_env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/homebrew/Caskroom/miniconda/base/envs/ds_env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/homebrew/Caskroom/miniconda/base/envs/ds_env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/opt/homebrew/Caskroom/miniconda/base/envs/ds_env/lib/python3.10/site-packages/sklearn/linear_model/_linear_los

## Cierre conceptual

La comparación no busca demostrar que los resultados numéricos sean idénticos línea por línea. Busca mostrar que la estructura matemática es la misma:

- Regresión lineal: `WX + b` con activación identidad y costo MSE.
- Regresión logística: `WX + b` con activación sigmoide y costo de entropía cruzada binaria.
- Perceptrón de una capa: estructura mínima que permite expresar ambos modelos y entrenarlos con descenso del gradiente.

Esta es la conexión importante antes de pasar a redes neuronales con capas ocultas.